In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib.patches import Patch # For legend

# ============================================
# Part 1: TSV 파일 병합 (효율성 개선)
# ============================================

def extract_column_name(filename):
    """
    파일명에서 stage-tissue-replicate 형식의 컬럼명 추출
    예: PRJNA400602_XENLAtx_SRR5988444_st14_NBa_B100T1.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv
    -> st14-NBa-1
    """
    # 파일명에서 st##_XXX_B###T## 패턴 추출
    pattern = r'(st\d+)_([A-Za-z0-9]+)_B\d+T\d+'
    match = re.search(pattern, filename)
    pattern2 = r'(st\d+)_([A-Za-z0-9]-[A-Za-z0-9]+)_B\d+T\d+'
    match2 = re.search(pattern2, filename)
    if match:
        stage = match.group(1)
        tissue = match.group(2)
        return stage, tissue
    elif match2:
        stage = match2.group(1)
        tissue = match2.group(2)
        return stage, tissue

    return None, NoneApple

def merge_tsv_files_efficiently(folder_path, output_csv='merged_tpm_data.csv'):
    """
    폴더 내 모든 TSV 파일을 효율적으로 병합 (pd.concat 사용)
    """
    tsv_files = [f for f in os.listdir(folder_path) if f.endswith('.tsv')]
    
    if not tsv_files:
        print("TSV 파일을 찾을 수 없습니다.")
        return None
    
    print(f"총 {len(tsv_files)}개의 TSV 파일을 발견했습니다.")
    
    dfs_to_concat = [] # 데이터프레임을 저장할 리스트
    column_counter = {}
    
    for tsv_file in tsv_files:
        file_path = os.path.join(folder_path, tsv_file)
        
        try:
            df = pd.read_csv(file_path, sep='\t', usecols=["Gene ID", "TPM"])
        except ValueError:
            print(f"경고: {tsv_file}에서 'Gene ID' 또는 'TPM' 컬럼을 찾을 수 없어 건너뜁니다.")
            continue
            
        if 'Gene ID' not in df.columns or 'TPM' not in df.columns:
            print(f"경고: {tsv_file}에 'Gene ID' 또는 'TPM' 컬럼이 없습니다.")
            continue
        
        df = df[['Gene ID', 'TPM']].copy()
        
        # Gene ID에서 'Gene ID:' 제거
        df['Gene ID'] = df['Gene ID'].astype(str).str.replace('Gene ID:', '', regex=False)
        
        # 컬럼명 생성
        stage, tissue = extract_column_name(tsv_file)
        if stage and tissue:
            key = f"{stage}-{tissue}"
            column_counter[key] = column_counter.get(key, 0) + 1
            new_column_name = f"{key}-{column_counter[key]}"
        else:
            new_column_name = tsv_file.replace('.tsv', '')
        
        # TPM 컬럼명 변경
        df = df.rename(columns={'TPM': new_column_name})
        
        # *** 효율성 개선 (1/2) ***
        # Gene ID를 인덱스로 설정하여 리스트에 추가
        df = df.set_index('Gene ID') 
        dfs_to_concat.append(df)
        
        print(f"처리 완료: {tsv_file} -> {new_column_name}")

    
    if not dfs_to_concat:
        print("병합할 데이터가 없습니다.")
        return None
    
    # *** 효율성 개선 (2/2) ***
    # pd.concat을 사용하여 모든 데이터프레임을 한 번에 병합 (axis=1: 컬럼 기준)
    # join='outer'는 모든 파일에 존재하는 모든 유전자를 포함시킵니다.
    print("\n데이터프레임 병합 중 (pd.concat)...")
    merged_df = pd.concat(dfs_to_concat, axis=1, join='outer')
    # Remove rows where Gene ID contains 'STRG'
df_filtered = df[~df['Gene ID'].str.contains('STRG', na=False)]

print(f"Original data shape: {df.shape}")
print(f"Filtered data shape: {df_filtered.shape}")
print(f"Removed {df.shape[0] - df_filtered.shape[0]} rows containing 'STRG'")

    # 인덱스(Gene ID)를 다시 컬럼으로 변환
    merged_df = merged_df.reset_index().rename(columns={'index': 'Gene ID'})
    
    # CSV 파일로 저장
    merged_df.to_csv(output_csv, index=False)
    print(f"\n병합 완료! 저장된 파일: {output_csv}")
    print(f"총 {len(merged_df)} 개의 유전자, {len(merged_df.columns)-1} 개의 샘플")
    
    return merged_df


# ============================================
# Part 2: PCA 분석 (효율성 개선)
# ============================================

def perform_pca_analysis_efficiently(csv_file='merged_tpm_data.csv', n_components=2):
    """
    병합된 CSV 파일을 불러와서 PCA 분석 수행 (메모리 효율성 개선)
    """
    # CSV 파일 읽기
    df = pd.read_csv(csv_file)
    print(f"\n데이터 로드 완료: {df.shape[0]} genes x {df.shape[1]-1} samples")
    # Remove rows where Gene ID contains 'STRG'
df_filtered = df[~df['Gene ID'].str.contains('STRG', na=False)]

print(f"Original data shape: {df.shape}")
print(f"Filtered data shape: {df_filtered.shape}")
print(f"Removed {df.shape[0] - df_filtered.shape[0]} rows containing 'STRG'")

    # Gene ID를 인덱스로 설정 (행: 유전자, 열: 샘플)
    df = df.set_index('Gene ID')
    
    # 결측치 처리 (0으로 채움)
    df = df.fillna(0)
    
    # *** 효율성 개선 (1/2) ***
    # 전치(transpose)하기 *전에* 유전자 필터링
    # (유전자 x 샘플) 형태에서 유전자 평균 계산 (axis=1)
    gene_means = df.mean(axis=1)
    genes_to_keep = gene_means[gene_means >= 1].index
    df_filtered = df.loc[genes_to_keep]
    print(f"필터링 후: {len(genes_to_keep)} 유전자 사용")

    # 로그 변환 (log2(TPM + 1))
    df_log = np.log2(df_filtered + 1)
    
    # *** 효율성 개선 (2/2) ***
    # 필터링 및 로그 변환이 끝난 *작은* 데이터프레임을 전치
    # (행: 샘플, 열: 유전자)
    df_transposed = df_log.T
    
    # 표준화 (Standardization)
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df_transposed)
    
    # PCA 수행
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(df_scaled)
    
    # 결과를 데이터프레임으로 변환
    pca_df = pd.DataFrame(
        pca_result,
        columns=[f'PC{i+1}' for i in range(n_components)],
        index=df_transposed.index # 인덱스는 샘플명
    )
    
    # 설명된 분산 출력
    print(f"\n설명된 분산:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var*100:.2f}%")
    
    # --- 이하 플롯 생성 코드는 원본과 동일 ---
    
    plt.figure(figsize=(10, 8))
    
    # Stage-tissue 정보 추출하여 색상 구분
    sample_info = ['-'.join(s.split('-')[:-1]) for s in pca_df.index]
    unique_groups = sorted(list(set(sample_info))) # 정렬하여 일관된 색상 매핑
    colors = sns.color_palette('husl', n_colors=len(unique_groups))
    color_map = {group: colors[i] for i, group in enumerate(unique_groups)}
    sample_colors = [color_map[info] for info in sample_info]
    
    plt.scatter(pca_df['PC1'], pca_df['PC2'], c=sample_colors, s=100, alpha=0.7)
    
    # 샘플 이름 표시
    for i, sample in enumerate(pca_df.index):
        plt.annotate(sample, (pca_df.loc[sample, 'PC1'], pca_df.loc[sample, 'PC2']), 
                     fontsize=8, alpha=0.7)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
    plt.title('PCA of TPM Expression Data')
    plt.grid(True, alpha=0.3)
    
    # 범례 추가
    legend_elements = [Patch(facecolor=color_map[group], label=group) 
                       for group in unique_groups]
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig('pca_plot.png', dpi=300, bbox_inches='tight')
    print(f"\nPCA 플롯 저장: pca_plot.png")
    plt.show()
    
    # PCA 결과 저장
    pca_df.to_csv('pca_results.csv')
    print(f"PCA 결과 저장: pca_results.csv")
    
    return pca_df, pca

# --- 실행 예시 ---
# if __name__ == '__main__':
#     # Part 1 실행
#     merged_data = merge_tsv_files_efficiently(folder_path='.')
    
#     # Part 2 실행
#     if merged_data is not None:
#         perform_pca_analysis_efficiently(csv_file='merged_tpm_data.csv')

In [ ]:
 perform_pca_analysis_efficiently(csv_file='merged_tpm_data.csv')

In [ ]:
if __name__ == '__main__':
     # Part 1 실행
    #merged_data = merge_tsv_files_efficiently(folder_path='/media/seungyun/8TBHardDisk1/XL_Plou2017/hisat2-tsv/st12/')
    
     # Part 2 실행
    if merged_data is not None:
        perform_pca_analysis_efficiently(csv_file='merged_tpm_data.csv')

총 22개의 TSV 파일을 발견했습니다.
처리 완료: PRJNA400602_XENLAtx_SRR5988459_st12_NPp_B108T50.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPp-1
처리 완료: PRJNA400602_XENLAtx_SRR5988471_st12_NPa_B108T74.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPa-1
처리 완료: PRJNA400602_XENLAtx_SRR5988468_st12_NBp_B108T73.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NBp-1
처리 완료: PRJNA400602_XENLAtx_SRR5988467_st12_NPp_B100T16.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPp-2
처리 완료: PRJNA400602_XENLAtx_SRR5988463_st12_NPa_B108T52.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPa-2
처리 완료: PRJNA400602_XENLAtx_SRR5988470_st12_NNE_B108T6.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NNE-1
처리 완료: PRJNA400602_XENLAtx_SRR5988469_st12_NBl_B108T53.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NBl-1
처리 완료: PRJNA400602_XENLAtx_SRR5988455_st12_NNE_B108T3.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NNE-2
처리 완료: PRJNA400602_XENLAtx_SRR5988462_st12_NNE_B108T70.xenLae10.hisat2.stri

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [4]:
folder_path = "/media/seungyun/8TBHardDisk1/XL_Plou2017/hisat2-tsv/st12"  # TSV 파일이 있는 폴더 경로
merged_df = merge_tsv_files(folder_path, output_csv='merged_tpm_data_st12.csv')
    
   




총 22개의 TSV 파일을 발견했습니다.
처리 완료: PRJNA400602_XENLAtx_SRR5988459_st12_NPp_B108T50.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPp-1
처리 완료: PRJNA400602_XENLAtx_SRR5988471_st12_NPa_B108T74.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPa-1
처리 완료: PRJNA400602_XENLAtx_SRR5988468_st12_NBp_B108T73.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NBp-1
처리 완료: PRJNA400602_XENLAtx_SRR5988467_st12_NPp_B100T16.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPp-2
처리 완료: PRJNA400602_XENLAtx_SRR5988463_st12_NPa_B108T52.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NPa-2
처리 완료: PRJNA400602_XENLAtx_SRR5988470_st12_NNE_B108T6.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NNE-1
처리 완료: PRJNA400602_XENLAtx_SRR5988469_st12_NBl_B108T53.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NBl-1
처리 완료: PRJNA400602_XENLAtx_SRR5988455_st12_NNE_B108T3.xenLae10.hisat2.stringtie_xb2025-01.tpm.tsv -> st12-NNE-2
처리 완료: PRJNA400602_XENLAtx_SRR5988462_st12_NNE_B108T70.xenLae10.hisat2.stri

In [3]:
 # Part 2: PCA 분석
#if merged_df is not None:
pca_df, pca_model = perform_pca_analysis('merged_tpm_data_st17.csv', n_components=2)

NameError: name 'perform_pca_analysis' is not defined